[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **Smart City Infrastructure Analysis**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

## Overview

This notebook demonstrates an **end-to-end computer vision pipeline** for **Automated Land Use and Land Cover (LULC) Classification** using **high-altitude drone imagery**. The workflow utilizes a **fine-tuned YOLO 11 Segmentation model** to automate the identification of **urban and rural features** from aerial perspectives.

---

## Real-World Applications

**Urban Planning**  
Automated mapping of building footprints and infrastructure density.

**Precision Agriculture**  
Monitoring farm boundaries and distinguishing managed crops from natural greenery.

**Environmental Monitoring**  
Tracking deforestation, sand encroachment, and vegetation health.

**Disaster Management**  
Real-time terrain analysis for flood or landslide risk assessment.

**Infrastructure Mapping**  
Identifying and segmenting major road networks for transportation analysis.

---

##  Key Technical Features

**High-Definition Segmentation**  
Implementation of **Retina Masks** to ensure **pixel-perfect boundary detection** for small-scale land features.

**Clean Visualization**  
Optimized inference logic to produce **"Pure Maps"** by removing **bounding boxes, labels, and confidence scores** for professional reporting.

**Optimized Inference**  
High-speed processing capable of **real-time performance** on industrial drone footage using a **Tesla T4 GPU**.

**High Accuracy**  
Achieved a **0.94 mAP** across **five distinct land classes**, ensuring reliable classification even in complex, varied terrains.

## Annotate your Custom dataset using Labellerr

 ***1. Visit the [Labellerr](https://www.labellerr.com/?utm_source=githubY&utm_medium=social&utm_campaign=github_clicks) website and click **“Sign Up”**.*** 

 ***2. After signing in, create your workspace by entering a unique name.***

 ***3. Navigate to your workspace’s API keys page (e.g., `https://<your-workspace>.labellerr.com/workspace/api-keys`) to generate your **API Key** and **API Secret**.***

 ***4. Store the credentials securely, and then use them to initialise the SDK or API client with `api_key`, `api_secret`.*** 



In [1]:
!git clone https://github.com/Labellerr/yolo_finetune_utils.git

Cloning into 'yolo_finetune_utils'...


## 🎞️ Random Frame Extraction from Video

Extracts a fixed number of high-quality frames from one or more videos to create an image dataset for annotation and training.

### 🔹 Purpose
- Convert raw manufacturing videos into individual image frames  
- Perform random sampling to avoid frame bias  
- Prepare data for annotation and YOLO training  


In [ ]:
from yolo_finetune_utils.frame_extractor import extract_random_frames

extract_random_frames(
    paths=['city map dataset.mp4'],
    total_images=100,
    out_dir="dataset_frames",
    jpg_quality=100,
    seed=42
)

[✓] Extracted 20 frames to folder: dataset_frames


## 📥 Download Annotations from Labellerr

After completing data labeling on the **Labellerr** platform, export the annotations in **COCO JSON format**.

Download the COCO JSON file from the Labellerr website and upload it into this project workspace to use it for further dataset preparation and training.

This COCO JSON file will be used in the next steps for:
- Frame–annotation alignment
- COCO → YOLO format conversion
- Model training and evaluation


# COCO to YOLO Format Conversion

Converts COCO-style segmentation annotations to YOLO segmentation dataset format.  
- Requires: `annotation.json` and images in `frames_output` directory.
- Output: Generated YOLO dataset folder.
- Parameters: allows train/val split, shuffling, and verbose mode.


In [3]:
# Import the converter utility
from yolo_finetune_utils.coco_yolo_converter.seg_converter import coco_to_yolo_converter

# Execute the conversion
coco_to_yolo_converter(
    json_path="01fafe75-c55f-432b-9e53-cb7d56fe8942.json",              # Your COCO file
    images_dir="dataset_frames",    # Where your 20 tiled images are
    output_dir="data/processed",                 # This will create 'labels' folder automatically
    use_split=True,                              # Split into Train/Val/Test
    train_ratio=0.8,
    val_ratio=0.2,                               # Increased for small dataset
    test_ratio=0.0,                              # Set to 0 since we only have 20 images
    shuffle=True,
    verbose=True
)

Conversion complete. Stats: {'train': 16, 'val': 4, 'test': 0}


{'stats': {'train': 16, 'val': 4, 'test': 0}, 'output_dir': 'data/processed'}

# Load and Train YOLO Segmentation Model

Loads the YOLO segmentation model and trains it using the converted YOLO dataset.
- Data: Path to YOLO-style `data.yaml`
- Parameters: epochs, image size, batch size, device, dataloader workers, experiment name.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt") 

# Training with heavy augmentation for a small dataset
model.train(
    data="data\processed\data.yaml",
    epochs=100,
    imgsz=640,
    # Augmentation Settings:
    degrees=15.0,    # Random rotation (+/- 15 degrees)
    flipud=0.5,      # Flip upside down (Great for drone shots!)
    fliplr=0.5,      # Flip left to right
    mosaic=1.0,      # Combine 4 images into 1 (Excellent for small data)
    mixup=0.1,       # Blend two images together
    hsv_h=0.015,     # Random color/hue adjustments
    scale=0.5        # Randomly zoom in/out
)

## Inference Configuration

This code block executes the **final model on the drone footage**, specifically configured to generate a **Professional Land-Use Map**.

---

### Model Loading
Initializes the **fine-tuned YOLO 11 weights (`best.pt`)** optimized for **aerial segmentation**.

### Precision Rendering (`retina_masks=True`)
Forces the model to generate **masks at the original video resolution**, ensuring **sharp, high-definition boundaries** for roads and structures.

### Clean Map UI
The parameters `show_boxes=False`, `show_conf=False`, and `show_labels=False` **strip away all technical metadata** (rectangles and text), leaving only the **color-coded land regions**.

### Sensitivity (`conf=0.4`)
Sets a **tailored confidence threshold** to maintain **stable segmentation across different flight altitudes and lighting conditions**.

### Output Management
Organizes the processed video into a **structured directory (`drone_map_output`)** for **easy retrieval and further analysis**.

In [ ]:
from ultralytics import YOLO

# 1. Load your best trained model
model = YOLO('/kaggle/working/runs/segment/train5/weights/best.pt')

# 2. Use the path you verified earlier
video_path = '/kaggle/input/datasets/aaryanaggarwal5040/drone-test/city map dataset.mp4'

# 3. Run inference with all "Detection UI" elements removed
results = model.predict(
    source=video_path,
    save=True,
    conf=0.4,
    imgsz=640,
    retina_masks=True,  # Keeps boundaries sharp
    show_boxes=False,   # Removes the rectangles
    show_conf=False,    # Removes the confidence percentages (0.85, etc.)
    show_labels=True,   # Keeps only the names (Building, Farm, etc.)
    project='/kaggle/working/runs/segment',
    name='drone_map_output'
)

print("✅ Professional Land-Map inference complete. No boxes or conf scores.")

---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
